### Mapping Uniport IDs to STRING IDs 

Loading files, make sure to change your directory to wherever you have cloned the 'yeast' github repo

In [ ]:
import polars as pl

In [ ]:
#Reading STRING Data 
file_path = r"..\..\data\4932.protein.links.detailed.v12.0.txt"
df = pl.scan_csv(file_path, separator=" ")

file_path2 = r"..\..\data\4932.protein.physical.links.detailed.v12.0.txt"
df2 = pl.scan_csv(file_path2, separator=" ")

## Reading AF3 Data 
confident = r"..\..\data\summary_confidences.parquet"
confident = pl.scan_parquet(confident)

pairs = r"..\..\data\summary_pairs.parquet"
pairs = pl.scan_parquet(pairs)

In [ ]:
## Matching Unirpot IDs to STRING IDs from what was given in the `pairs` file
unique_protiens = (
    pl.concat([
        pairs.select(pl.col("af3_id1").alias("protein_id")),
        pairs.select(pl.col("af3_id2").alias("protein_id"))
    ])
    .drop_nulls()
    .unique()
    .sort("protein_id")
)

map = (
    pl.concat([
        pairs.select([
            pl.col("af3_id1").alias("uniprot_id"),
            pl.col("michaelis2023:Source Gene names (SGD/UniProt-primary or ordered locus)") #some of these are empty 
                .alias("gene_name"),
    ]),
    pairs.select([
     pl.col("af3_id2").alias("uniprot_id"),
     pl.col("michaelis2023:Target Gene names  (SGD/UniProt-primary or ordered locus)") #some of these are empty
        .alias("gene_name"),
        ])
    ])
    .drop_nulls()
    .unique(subset =["uniprot_id"])
)

# print(pairs.collect().head())

Now we will check how proteins we are able to map, and how many need to be mapped

In [ ]:
coverage = map.collect().height
total = unique_protiens.collect().height
print(f"Already mapped: {coverage}/{total} ({100 *coverage/total:.1f}%)")
#What we are able to map ^

already_mapped_ids = map.collect()["uniprot_id"].to_list()
need_api = unique_protiens.filter(
    pl.col("protein_id").is_in(already_mapped_ids).not_()
)
print(f"Need API lookup: {need_api.collect().height}") #missing IDs


Mapping the IDs together now 

In [ ]:
string_aliases = (
    pl.scan_csv("..\\..\\data\\4932.protein.aliases.v12.0.txt", separator="\t")
    .rename({"#string_protein_id": "string_id", "alias": "protein_id"})
    # Filter for UniProt to ensure clean, relevant mappings
    .with_columns(pl.col("protein_id").str.to_lowercase())
    .filter(pl.col("source").str.contains("UniProt"))
)

# Match api LazyFrame against the STRING aliases
# This maps your single-column missing IDs to their official STRING IDs
matched_missing = (
    need_api.join(
        string_aliases,
        on="protein_id",
        how="inner"
    )
    .select([
        pl.col("protein_id").alias("uniprot_id"),
        pl.col("string_id")
    ])
    .unique()
)

unique_protiens = (
    unique_protiens.join(
        string_aliases,
        on="protein_id",
        how="inner"
    )
    .select(
        pl.col("protein_id").alias("uniprot_id"),
        pl.col("string_id")
    )
)

string_info = (
    pl.scan_csv("..\\..\\data\\4932.protein.info.v12.0.txt", separator="\t")
    .rename({"#string_protein_id": "string_id"})
)

full_matches = (
    matched_missing.join(
        string_info,
        on="string_id",
        how="inner"
    )
    .select([
        pl.col("uniprot_id"),
        pl.col("string_id"),
        pl.col("preferred_name").alias("string_gene_name"),
        pl.col("annotation")
    ])
)

unique_protiens = (
    unique_protiens.join(
        string_info,
        on="string_id",
        how="inner"
    )
    .select([
    pl.col("uniprot_id"),
        pl.col("string_id"),
        pl.col("preferred_name").alias("string_gene_name"),
        pl.col("annotation")
    ])
)


# see your results
df_matches = full_matches.collect()
print(f"Found local STRING matches for {df_matches.height} of your missing IDs!")
print(df_matches.head())
print(len(df_matches))

Finding the duplicates in the STRING IDs

In [ ]:
## Find the duplicates in this data set 

non_dupe = []
dupe = []

protein_id = df_matches["uniprot_id"]
for i in protein_id:
    if i not in non_dupe:
        non_dupe.append(i)
    else: 
        dupe.append(i)


In [ ]:
## Before mapping them I want to add [string_id┆ string_gene_name ┆ annotation] columsn to map
## and I want to add [gene_name] to df_matches_unique/df_matches_unique_lazy

matched = (
    map.join(
        string_aliases,
        on="protein_id",
        how="inner"
    )
    .select([
        pl.col("protein_id").alias("uniprot_id"),
        pl.col("string_id")
    ])
    .unique()
)

string_info = (
    pl.scan_csv("..\\..\\data\\4932.protein.info.v12.0.txt", separator="\t")
    .rename({"#string_protein_id": "string_id"})
)

full_matches = (
    matched_missing.join(
        string_info,
        on="string_id",
        how="inner"
    )
    .select([
        pl.col("uniprot_id"),
        pl.col("string_id"),
        pl.col("preferred_name").alias("string_gene_name"),
        pl.col("annotation")
    ])
)

df_matches = full_matches.collect()
print(f"Found local STRING matches for {df_matches.height} of your missing IDs!")
print(df_matches.head())
print(len(df_matches)) 

Joining everything

In [ ]:
import polars.selectors as cs

df_matches_unique_lazy = df_matches.unique(subset=['uniprot_id'], keep='first').lazy()

df_final_lazy = unique_protiens.unique(subset=['uniprot_id'], keep='first').join(
    other=df_matches_unique_lazy,
    on='uniprot_id',
    how='left'
    ).select(pl.exclude("^.*_right$"))

df_final = df_final_lazy.collect()

print(f"Final DataFrame Shape: {df_final.shape}")
print(df_final.head())
print(len(df_final))

#### Adding the scores for ipTM and STRING
Remake the mapping table to see the protein-protein interactions with the uniprot id and string id, and then add there score in another column like `STRING_score` and `iptm_score`. Then make some sort of histogram so see the comparrison, but there will only have 5937 proteins instead of the 6038 used.

In [ ]:
## df2 <- physical links
## confident and pairs <- have the ipTM scores
df2_2 = (
    df2.select([
        pl.col("protein1"),
        pl.col("protein2"),
        pl.col("combined_score")
    ])
    .collect()
    .unique()
)
print("\nFirst 5 rows of df2_2:")
print(df2_2.head())
print(len(df2_2))

# Now to do this with pairs 
pairs_2 = (
    pairs.select([
        pl.col("af3_id1").alias("protein1"),
        pl.col("af3_id2").alias("protein2"),
        pl.col("chain_pair_iptm_best"),
        pl.col("chain_pair_iptm_mean"),
        pl.col("chain_pair_iptm_best_corrected"),
        pl.col("chain_pair_iptm_mean_corrected")
    ])
    .collect()
    .drop_nulls()
    .unique()
)
print(pairs_2.head())

In [ ]:
### Remap and visualize the scores 
# the uniprot_id amd value being the string_id then iterate and combine/map my two lazyframes to everything matches up

# Create a quick lookup dictionary: {uniprot_id: string_id}
id_map_dict = dict(zip(df_final['uniprot_id'], df_final['string_id']))

# using pairs_2 protein1 and protein2 to make a new map
df_alphafold_mapped = pairs_2.with_columns([
    pl.col("protein1").replace(id_map_dict, default=None).alias("string_id1"),
    pl.col("protein2").replace(id_map_dict, default=None).alias("string_id2")
])

# Drop any rows where the mapping returned None 
df_alphafold_mapped = df_alphafold_mapped.drop_nulls(subset=["string_id1", "string_id2"])
print(len(df_alphafold_mapped)) # ~ 7.5 mil instead of 7.6 mil as some proteins didn't map

In [ ]:
df_all_mapped = df_alphafold_mapped.with_columns(
    pl.min_horizontal("string_id1", "string_id2").alias("pair_key1"),
    pl.max_horizontal("string_id1", "string_id2").alias("pair_key2")
)

df_string_ordered = df2_2.with_columns(
    pl.min_horizontal("protein1", "protein2").alias("pair_key1"),
    pl.max_horizontal("protein1", "protein2").alias("pair_key2")
)

df_string_unique_pairs = df_string_ordered.select(
    ["pair_key1", "pair_key2", "combined_score"]
).unique(subset=["pair_key1", "pair_key2"])

df_final_comparison = df_all_mapped.join(
    df_string_unique_pairs,
    on=["pair_key1", "pair_key2"],
    how="left"
).drop("pair_key1", "pair_key2")

print(df_final_comparison.head())
print(len(df_final_comparison)) # Finished adding STRING Scorces